In [1]:
# import sys
# sys.path.append('../src')

import sys
sys.path.append('../src')
from pyomo.environ import value as pyoval

In [2]:
import pandas as pd
import numpy
from energiapy.components.temporal_scale import TemporalScale
from energiapy.components.resource import Resource, VaryingResource, Scope
from energiapy.components.process import Process, VaryingProcess
from energiapy.components.material import Material
from energiapy.components.location import Location
from energiapy.components.network import Network
from energiapy.components.scenario import Scenario
from energiapy.components.transport import Transport
from energiapy.model.formulate import formulate, Constraints, Objective
from energiapy.utils.nsrdb_utils import fetch_nsrdb_data
from energiapy.plot import plot_results, plot_scenario
from energiapy.plot.plot_results import CostY, CostX
from energiapy.model.solve import solve
from itertools import product
from energiapy.components.result import Result
from itertools import product
from matplotlib import rc
import matplotlib.pyplot as plt
from energiapy.model.solve import solve
from energiapy.plot.plot_results import CostY, CostX
from energiapy.components.location import Location
from energiapy.plot import plot_results, plot_scenario
from energiapy.utils.nsrdb_utils import fetch_nsrdb_data
from energiapy.model.formulate import formulate, Constraints, Objective
from energiapy.model.bounds import CapacityBounds
from energiapy.utils.data_utils import get_data, make_henry_price_df, remove_outliers, load_results
from energiapy.conversion.photovoltaic import solar_power_output
from energiapy.conversion.windmill import wind_power_output
from energiapy.model.constraints.integer_cuts import constraint_block_integer_cut, constraint_block_integer_cut_min


# import pandas as pd
# import numpy
# from energiapy.components.temporal_scale import TemporalScale
# from energiapy.components.resource import Resource, VaryingResource, Scope
# from energiapy.components.process import Process, VaryingProcess
# from energiapy.components.material import Material
# from energiapy.components.location import Location
# from energiapy.components.network import Network
# from energiapy.components.scenario import Scenario
# from energiapy.components.transport import Transport
# from energiapy.model.formulate import formulate, Constraints, Objective
# from energiapy.utils.nsrdb_utils import fetch_nsrdb_data
# from energiapy.plot import plot_results, plot_scenario
# from energiapy.plot.plot_results import CostY, CostX
# from energiapy.model.solve import solve
# from itertools import product
# from energiapy.components.result import Result
# from itertools import product
# from matplotlib import rc
# import matplotlib.pyplot as plt
# from energiapy.model.solve import solve
# from energiapy.plot.plot_results import CostY, CostX
# from energiapy.components.location import Location
# from energiapy.plot import plot_results, plot_scenario
# from energiapy.utils.nsrdb_utils import fetch_nsrdb_data
# from energiapy.model.formulate import formulate, Constraints, Objective
# from energiapy.model.bounds import CapacityBounds
# from energiapy.utils.data_utils import get_data, make_henry_price_df, remove_outliers, load_results
# from energiapy.conversion.photovoltaic import solar_power_output
# from energiapy.conversion.windmill import wind_power_output
# from energiapy.model.constraints.integer_cuts import constraint_block_integer_cut, constraint_block_integer_cut_min
# from scipy import stats

In [3]:
scales = TemporalScale(discretization_list=[1, 365, 24])

In [4]:
bigM = 10**7
smallM = 0.001

Hourly Electricity Pricing

In [5]:
# # Read and process 2021 data
# edf1 = pd.read_excel("C:/Users/marcopdsousa/Documents/energiapy/AICHE/Historical electricity prices/2021 Historical electricity prices.xlsx", sheet_name=None)
# edf1 = pd.concat(edf1.values(), ignore_index=True)
# edf1 = edf1[edf1['Settlement Point'] == 'HB_HOUSTON']['Settlement Point Price'].reset_index(drop = True)

# # Read and process 2022 data
# edf2 = pd.read_excel("C:/Users/marcopdsousa/Documents/energiapy/AICHE/Historical electricity prices/2022 Historical electricity prices.xlsx", sheet_name=None)
# edf2 = pd.concat(edf2.values(), ignore_index=True)
# edf2 = edf2[edf2['Settlement Point'] == 'HB_HOUSTON']['Settlement Point Price'].reset_index(drop = True)

# Read and process 2023 data
edf3 = pd.read_excel("C:/Users/marcopdsousa/Documents/energiapy/AICHE/Historical electricity prices/2023 Historical electricity prices.xlsx", sheet_name=None)
edf3 = pd.concat(edf3.values(), ignore_index=True)
edf3 = edf3[edf3['Settlement Point'] == 'HB_HOUSTON']['Settlement Point Price'].reset_index(drop = True)

ecombined_df = pd.DataFrame(edf3)
# ecombined_df

# # Calculate the mean and standard deviation
mean = ecombined_df.mean()
std = ecombined_df.std()

# mean = edf3.mean()
# std = edf3.std()


outliers = numpy.abs(ecombined_df - mean) > 2 * std
# Prune outliers beyond 3 standard deviations
ecombined_df = ecombined_df.mask(outliers)
fill_value = ecombined_df.mean()
ecombined_df = ecombined_df.fillna(fill_value)
ecombined_df

# Get the maximum value from the combined data
#combined = ecombined_df.max()

# Output the maximum value
#print(ecombined)

,Settlement Point Price
0,10.37
1,9.85
2,8.58
3,8.18
4,8.33
...,...
8755,19.29
8756,16.57
8757,14.47
8758,15.50


Historical Data for Wind Availability

In [6]:
# wfdf1 = pd.read_excel("C:/Users/marcopdsousa/Documents/energiapy/AICHE/Historical Renewable availability/2021 Wind and Solar data.xlsx", sheet_name='Wind Data')  # None reads all sheets
# wfdf1 = wfdf1['ERCOT.WIND.GEN']

# wfdf2 = pd.read_excel("C:/Users/marcopdsousa/Documents/energiapy/AICHE/Historical Renewable availability/2022 Wind and Solar data.xlsx", sheet_name='Wind Data')  # None reads all sheets
# wfdf2 = wfdf2['ERCOT.WIND.GEN']

wfdf3 = pd.read_excel("C:/Users/marcopdsousa/Documents/energiapy/AICHE/Historical Renewable availability/2023 Wind and Solar data.xlsx", sheet_name='Wind Data')  # None reads all sheets
wfdf3 = wfdf3['ERCOT.WIND.GEN']
wfdf3

wfcombined_df = pd.concat([wfdf3], axis=0)  # axis=0 means row-wise (default)
wfcombined_df = pd.DataFrame(wfcombined_df)
#wfdf1, wfdf2, 
wfcombined_df

,ERCOT.WIND.GEN
0,21753.284477
1,21570.395694
2,21036.504013
3,20595.481515
4,20387.843555
...,...
8755,17161.333953
8756,18300.677938
8757,18607.202018
8758,17865.765282


Solar Availability

In [7]:
# sfdf1 = pd.read_excel("C:/Users/marcopdsousa/Documents/energiapy/AICHE/Historical Renewable availability/2021 Wind and Solar data.xlsx", sheet_name='Solar Data')  # None reads all sheets
# sfdf1 = sfdf1['ERCOT.PVGR.GEN']

# sfdf2 = pd.read_excel("C:/Users/marcopdsousa/Documents/energiapy/AICHE/Historical Renewable availability/2022 Wind and Solar data.xlsx", sheet_name='Solar Data')  # None reads all sheets
# sfdf2 = sfdf2['ERCOT.PVGR.GEN']

sfdf3 = pd.read_excel("C:/Users/marcopdsousa/Documents/energiapy/AICHE/Historical Renewable availability/2023 Wind and Solar data.xlsx", sheet_name='Solar Data')  # None reads all sheets
sfdf3 = sfdf3['PV_Final']
sfcombined_df = pd.concat([sfdf3], axis=0)  # axis=0 means row-wise (default)
sfcombined_df = pd.DataFrame(sfcombined_df)
sfcombined_df
#sfdf1, sfdf2, 

,PV_Final
0,0.0
1,0.0
2,0.0
3,0.0
4,0.0
...,...
8755,0.0
8756,0.0
8757,0.0
8758,0.0


In [8]:
# sfcombined_df[:24]

In [9]:
# wind_pred = pd.read_csv('WindPred.csv')
# demand_pred = pd.read_csv('DemandPred.csv')
# demand_pred = demand_pred.set_index('Unnamed: 0')
# wind_pred = wind_pred.set_index('Unnamed: 0')

In [10]:
ecombined = ecombined_df.max()
wfcombined = wfcombined_df.max()
sfcombined = sfcombined_df.max()

In [11]:
Wind = Resource(name='Wind', cons_max = bigM, basis='MW', price = 60.11, label='Wind')
Solar = Resource(name='Solar', basis='MW', cons_max = bigM, label='Solar', price= 52.69)
Power = Resource(name='Power', basis='MW', label='Power')  #, demand=True, varying=[VaryingResource.DETERMINISTIC_DEMAND]
GridPower = Resource(name='GridPower', cons_max= bigM, basis='MW',label='GridPower', price=ecombined, varying = [VaryingResource.DETERMINISTIC_PRICE])
# ecombined, varying = [VaryingResource.DETERMINISTIC_PRICE]
#Reactor Streams
Ethane_feed = Resource(name='Ethane_feed', price=0.2 * 30.07/1000, basis='mol/hr',cons_max=bigM, label='Ethane_feed') #converted cost to mole basis

Ethylene_rout = Resource(name='Ethylene_rout', basis='mol/hr', label='Ethylene_rout')
H2_rout = Resource(name='H2_rout', basis='mol/hr', label='H2_rout')
Methane_rout = Resource(name='Methane_rout', basis='mol/hr', label='Methane_rout')
Ethane_rout = Resource(name='Ethane_rout', basis='mol/hr', label='Ethane_rout')


#Separator outlet Streams

H2_sepout = Resource(name='H2_sepout', basis='mol/hr', label='H2_sepout')
Methane_sepout = Resource(name='Methane_sepout', basis='mol/hr', label='Methane_sepout') #Dummy GWP
Ethylene_pure = Resource(name='Ethylene_pure', basis='mol/hr', label='Ethylene_pure',demand=True) #Shoule I put sell=true here?
Ethane_recycle = Resource(name='Ethane_recycle', basis='mol/hr', label='Ethane recycle from columns',sell=True)

#PSA OUtLET
H2_pure = Resource(name='H2_pure', revenue= 1.06 * 2.016/1000, basis='mol/hr', label='H2_pure',sell = True) 
Methane_pure = Resource(name='Methane_pure', basis='mol/hr', gwp=1, label='Methane_pure', sell=True) 

In [12]:
LiI = Process(name='LiI', storage=Power, vopex=1302, capex=35164, fopex=0, prod_min=0,  prod_max=200,
              storage_loss=0.11,  label='Lithium-ion battery', basis='MW', store_max=bigM) #Need to verify CAPEX, OPEX with Harsh. Kept his numbers

Grid = Process(name='Grid', conversion={GridPower: -1, Power: 1}, prod_min=smallM, prod_max=bigM, label='Grid', basis='MW') 

PV = Process(name='PV', conversion={Solar: -1, Power: 1},
             prod_max=150, prod_min=150,  varying= [VaryingProcess.DETERMINISTIC_CAPACITY], label='Solar PV', basis='MW')

WF = Process(name='WF', conversion={Wind: -1/0.85, Power: 1}, vopex=990, capex=146200, fopex=0, prod_min=300,
             prod_max=300, varying=[VaryingProcess.DETERMINISTIC_CAPACITY], label='Wind mill array', basis='MW') #Kept Harsh's numbers. Need to verify

Mreact = Process(name='Mreact', conversion={Ethylene_rout: 1, H2_rout: 1.185185185, Ethane_rout:  0.465763638, Methane_rout: 0.259259259, Ethane_feed: - 1.254293305, Power: -1},\
    capex= 107803.5466, fopex=0.04*0.730, vopex=0.057434048, prod_min=smallM, prod_max=bigM, label='Mreact') #OPEX need to be entered

Sep = Process(name='Sep', conversion = {Ethylene_pure: 1, H2_sepout: 1.1946285081, Methane_sepout: 0.2613249861 , Ethane_recycle: .4687512463, \
    Ethane_rout: -0.4694747511, Ethylene_rout: -1.0079678037, Methane_rout: -0.2613249861, H2_rout: -1.1946285081, Power: -1},\
        capex = 6622.12417, vopex =  0.066174012, fopex = 0.1, label='Sep')

psa = Process(name='psa', conversion = {H2_sepout: -1, H2_pure: 1, Methane_sepout: -0.21875, Methane_pure: 0.21875, Power:-0.5}, capex=4522.2104, fopex= 0, vopex= 0.02) #Not finalized VOPEX

Turbine_simple = Process(name='Turbine_simple', conversion = {H2_pure: -1/0.3, Power: 1}, basis= 'MW', capex = 835.5/1000, fopex = 6.87/1000, vopex = 1.24/1000, gwp = 0, label='Turbine_simple')

Turbine_combined = Process(name='Turbine_combined', conversion = {H2_pure: -1/0.6, Power: 1}, basis= 'MW', capex = 920.9/1000, fopex = 15.51/1000, vopex = 3.33/1000, gwp=0, label='Turbine_combined')

In [13]:
process_set = { PV, WF, Grid, Mreact,Sep,psa}

In [14]:
houston = Location(name='HO', processes=process_set, capacity_factor={PV: sfcombined_df, WF: wfcombined_df}, price_factor={GridPower: ecombined_df}, 
                   scales=scales, label='Houston', expenditure_scale_level=0, demand_scale_level=1, price_scale_level=2, capacity_scale_level=2)

#, price_factor={GridPower: ecombined_df

In [15]:

scenario = Scenario(name='scenario_full', network=houston, scales=scales,  expenditure_scale_level=0, demand_scale_level=1, 
                    purchase_scale_level = 2, scheduling_scale_level=2, network_scale_level=0, capacity_scale_level=2,
                    label='full_case', demand={houston: {Ethylene_pure:100}}, annualization_factor = 1/25) #Demand constraint at hourly level. Annualization factor would be specified here.

In [16]:
milp_cost = formulate(scenario=scenario, constraints={Constraints.COST, Constraints.INVENTORY, Constraints.NETWORK, Constraints.PRODUCTION,
                      Constraints.RESOURCE_BALANCE, Constraints.DEMAND}, objective=Objective.COST, demand_sign = 'geq')

constraint process capex
constraint process fopex
constraint process vopex
constraint storage capex
constraint process incidental
constraint location capex
constraint storage cost
constraint storage cost location
constraint storage cost network
constraint production mode
constraint inventory balance
ERROR: Rule failed when generating expression for Constraint
constraint_resource_purchase_varying with index ('HO', 'GridPower', 0, 0, 0):
AttributeError: 'Series' object has no attribute 'is_component_type'
ERROR: Constructing component 'constraint_resource_purchase_varying' from
data=None failed:
        AttributeError: 'Series' object has no attribute 'is_component_type'


AttributeError: 'Series' object has no attribute 'is_component_type'

In [ ]:
# from pyomo.environ import Objective

# list(milp_cost.component_objects(Objective, active=True))

# for obj in milp_cost.component_objects(Objective, active=True):
#     print("Objective name:", obj.name)
#     obj.pprint()


In [ ]:
results_cost = solve(scenario=scenario, instance=milp_cost,
                     solver='gurobi', name=f"res_cost", print_solversteps=True, saveformat = '.pkl')